# RutaPro · Notebook 07: Evidencia para la matriz de afinidad (reglas del simulador)

**Problema.** La matriz **objetivo laboral × categoría de curso** es la regla principal del simulador (peso 2.0 en la utilidad), y en el v1 es "un supuesto del equipo basado en lo que suelen pedir los avisos de prácticas". Es una afirmación sin evidencia visible: cualquier jurado puede preguntar *"¿de dónde salen esos números?"*.

**Qué hace este notebook.** Convierte ese supuesto en una **estimación basada en avisos reales** de prácticas:
1. Se tabulan a mano avisos reales (protocolo abajo) en `01_Datos/evidencia_avisos.csv`.
2. Se calcula la afinidad empírica de cada puesto × categoría con intervalos de confianza (bootstrap sobre avisos).
3. Se compara con la matriz supuesta: correlación, error medio y celdas donde el supuesto queda fuera del intervalo.
4. Se **re-simula** con la matriz empírica y se verifica si las conclusiones de los notebooks 04 a 06 se sostienen.

> **Importante.** Este notebook necesita un archivo que el equipo debe construir. **No contiene datos inventados.** Si el archivo no existe, el notebook lo indica y se detiene sin error.

## Protocolo de recolección (manual)

- **Fuentes:** portales de empleo con avisos de prácticas preprofesionales en el Perú (por ejemplo Computrabajo, Bumeran, Indeed, LinkedIn). La recolección es **manual** (leer cada aviso y codificarlo); no se hace scraping, para respetar los términos de uso de cada portal.
- **Tamaño:** al menos **20 avisos por cada uno de los 6 puestos** (120 en total). Con menos, los intervalos serán demasiado anchos.
- **Selección:** buscar el puesto objetivo ("practicante de análisis de datos", "practicante BI", "practicante de ciencia de datos", "practicante financiero", "practicante de marketing analytics", "practicante de gestión de proyectos") y tomar los primeros avisos vigentes, sin escoger por su contenido. Registrar `url` y `fecha_consulta`.
- **Codificación** de cada una de las 11 categorías: `0` = no se menciona, `1` = deseable o "se valora", `2` = requisito obligatorio.
- **Calidad:** idealmente dos integrantes codifican los mismos avisos de forma independiente (archivo `evidencia_avisos_codificador2.csv`) y el notebook calcula el acuerdo (kappa de Cohen).

In [1]:
import os, sys, subprocess
from google.colab import drive

NECESITA_LIGHTFM = False

# 1) Conectar el Drive y entrar a la carpeta del proyecto
drive.mount('/content/drive')
PROY = "/content/drive/MyDrive/RutaPro"
os.chdir(PROY + "/02_Notebooks")
print("Carpeta actual:", os.getcwd())

# 2) Comprobar que están los archivos imprescindibles
faltan = [f for f in ["rutapro_rec.py", "rutapro_sim.py", "rutapro_extra.py", "rutapro_mooc.py",
                      "resultados/hiperparametros.json", "../01_Datos/cursos.csv",
                      "../01_Datos/pares_usuario_curso.csv"] if not os.path.exists(f)]
print("Archivos que faltan:", faltan if faltan else "ninguno, todo en su sitio")

# 3) LightFM: usa la copia compilada en tu Drive; si no existe, la compila (≈ 2 min) y la guarda
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:], r.stderr[-1500:])
        raise RuntimeError("Falló: " + cmd[:70])

if NECESITA_LIGHTFM:
    LF = PROY + "/lightfm_src"
    if not os.path.exists(LF + "/lightfm"):
        print("Compilando LightFM, un momento...")
        sh("pip install -q cython")
        sh("rm -rf /content/lightfm_src && git clone -q --depth 1 https://github.com/lyst/lightfm.git /content/lightfm_src")
        p = "/content/lightfm_src/setup.py"
        s = open(p).read()
        s = s.replace("compiler_directives={'language_level' : \"3\"}",
                      "compiler_directives={'language_level': \"3\", 'legacy_implicit_noexcept': True}").replace("-march=native", "-O2")
        open(p, "w").write(s)
        sh("cd /content/lightfm_src && python setup.py cythonize -q && python setup.py build_ext --inplace -q")
        sh("rm -rf '" + LF + "' && cp -r /content/lightfm_src '" + LF + "'")
    sys.path.insert(0, LF)
    import lightfm
    print("LightFM", lightfm.__version__)

Mounted at /content/drive
Carpeta actual: /content/drive/MyDrive/RutaPro/02_Notebooks
Archivos que faltan: ninguno, todo en su sitio


In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import stats
warnings.filterwarnings("ignore")

import rutapro_rec as rr
import rutapro_sim as sim
import rutapro_extra as ex

AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, GRIS = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#9a9890"
TXT, TXT2 = "#0b0b0b", "#52514e"
mpl.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#d4d3cd", "axes.labelcolor": TXT2, "xtick.color": TXT2, "ytick.color": TXT2,
    "text.color": TXT, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False, "font.size": 10, "figure.dpi": 110,
})
BASE = rr.ruta_base(); DATA = BASE / "01_Datos"
FIG = BASE / "04_Informe" / "figuras"; FIG.mkdir(parents=True, exist_ok=True)
RES = BASE / "02_Notebooks" / "resultados_robustez"; RES.mkdir(parents=True, exist_ok=True)
def guardar(fig, n): fig.savefig(FIG / f"{n}.png", dpi=160, bbox_inches="tight")

ARCH = DATA / "evidencia_avisos.csv"
ARCH2 = DATA / "evidencia_avisos_codificador2.csv"
PLANTILLA = DATA / "evidencia_avisos_plantilla.csv"
COLUMNAS = ["aviso_id", "fuente", "url", "fecha_consulta", "empresa", "puesto_objetivo"] + sim.CATS
pd.DataFrame(columns=COLUMNAS).to_csv(PLANTILLA, index=False, encoding="utf-8")
print("Plantilla escrita en", PLANTILLA.relative_to(BASE))
print("Puestos válidos para 'puesto_objetivo':", sim.OBJETIVOS)
print("Categorías (0 = no se menciona, 1 = deseable, 2 = requisito):", sim.CATS)
HAY_EVIDENCIA = ARCH.exists()
print("\nEvidencia cargada:", HAY_EVIDENCIA)
if not HAY_EVIDENCIA:
    print("No existe", ARCH.relative_to(BASE), "→ completa la plantilla con avisos reales y vuelve a ejecutar este notebook.")
    print("El resto de las celdas se omite sin error.")

Plantilla escrita en 01_Datos/evidencia_avisos_plantilla.csv
Puestos válidos para 'puesto_objetivo': ['Data Analyst', 'BI Analyst', 'Data Scientist', 'Analista Financiero', 'Marketing Analyst', 'Project Analyst']
Categorías (0 = no se menciona, 1 = deseable, 2 = requisito): ['Python', 'SQL y Bases de Datos', 'Excel', 'Power BI', 'Tableau y Visualización', 'Estadística', 'R', 'Machine Learning', 'Análisis de Datos', 'Gestión de Proyectos', 'Habilidades Blandas']

Evidencia cargada: False
No existe 01_Datos/evidencia_avisos.csv → completa la plantilla con avisos reales y vuelve a ejecutar este notebook.
El resto de las celdas se omite sin error.


## 1. Validación de la evidencia

In [ ]:
if HAY_EVIDENCIA:
    ev = pd.read_csv(ARCH)
    faltan = [c for c in COLUMNAS if c not in ev.columns]
    assert not faltan, f"Faltan columnas: {faltan}"
    assert ev["puesto_objetivo"].isin(sim.OBJETIVOS).all(), "puesto_objetivo con valores no válidos"
    assert ev[sim.CATS].isin([0, 1, 2]).all().all(), "Las categorías solo admiten 0, 1 o 2"
    assert ev["aviso_id"].is_unique, "aviso_id repetido"
    n_por_puesto = ev.groupby("puesto_objetivo").size().reindex(sim.OBJETIVOS, fill_value=0)
    display(n_por_puesto.to_frame("avisos"))
    pocos = n_por_puesto[n_por_puesto < 20]
    if len(pocos): print("ADVERTENCIA: menos de 20 avisos en", list(pocos.index), "→ los intervalos serán muy anchos.")
    print("Fuentes:", ev.fuente.value_counts().to_dict())

In [ ]:
if HAY_EVIDENCIA and ARCH2.exists():
    from sklearn.metrics import cohen_kappa_score
    e2 = pd.read_csv(ARCH2).set_index("aviso_id"); e1 = ev.set_index("aviso_id")
    comunes = e1.index.intersection(e2.index)
    kap = {c: cohen_kappa_score(e1.loc[comunes, c], e2.loc[comunes, c]) for c in sim.CATS}
    print(f"Acuerdo entre codificadores en {len(comunes)} avisos (kappa de Cohen por categoría):")
    display(pd.Series(kap).round(2).to_frame("kappa"))
    print("Orientación habitual: < 0.4 bajo, 0.4 a 0.6 moderado, 0.6 a 0.8 sustancial, > 0.8 casi perfecto.")
elif HAY_EVIDENCIA:
    print("No hay segundo codificador (opcional): sin acuerdo entre codificadores que reportar.")

## 2. Afinidad empírica y comparación con el supuesto

Afinidad empírica de un puesto en una categoría = promedio de `valor / 2` sobre los avisos de ese puesto (requisito = 1, deseable = 0.5, no mencionado = 0). Como la matriz supuesta tiene un máximo de 1.0 por puesto, se reporta también la versión **normalizada por fila** (la categoría más demandada de cada puesto vale 1), que es la comparable y la que se usa para re-simular. Los intervalos son bootstrap sobre avisos.

In [ ]:
if HAY_EVIDENCIA:
    rng = np.random.default_rng(0)
    emp, lo, hi = {}, {}, {}
    for p in sim.OBJETIVOS:
        X = ev.loc[ev.puesto_objetivo == p, sim.CATS].values / 2.0
        emp[p] = X.mean(axis=0)
        boot = np.array([X[rng.integers(0, len(X), len(X))].mean(axis=0) for _ in range(2000)])
        lo[p], hi[p] = np.percentile(boot, 2.5, axis=0), np.percentile(boot, 97.5, axis=0)
    EMP = pd.DataFrame(emp, index=sim.CATS).T; LO = pd.DataFrame(lo, index=sim.CATS).T; HI = pd.DataFrame(hi, index=sim.CATS).T
    EMP_N = EMP.div(EMP.max(axis=1), axis=0)                         # normalizada por fila
    SUP = sim.AFINIDAD.loc[sim.OBJETIVOS, sim.CATS]
    comp = pd.DataFrame({"supuesta": SUP.stack(), "empírica (normalizada)": EMP_N.stack(), "empírica (absoluta)": EMP.stack(),
                         "IC95 inf (abs.)": LO.stack(), "IC95 sup (abs.)": HI.stack()})
    comp["dif (emp. norm. − supuesta)"] = comp["empírica (normalizada)"] - comp["supuesta"]
    comp.to_csv(RES / "afinidad_comparacion.csv")
    EMP_N.to_csv(RES / "regla_afinidad_empirica.csv")
    rho = stats.spearmanr(comp["supuesta"], comp["empírica (normalizada)"])[0]
    mae = comp["dif (emp. norm. − supuesta)"].abs().mean()
    print(f"Correlación de Spearman entre las 66 celdas (supuesta vs empírica normalizada): {rho:.2f}")
    print(f"Error absoluto medio: {mae:.2f} | celdas con |dif| > 0.3: {(comp['dif (emp. norm. − supuesta)'].abs() > 0.3).sum()} de {len(comp)}")
    print("\nCeldas donde el supuesto se aleja más de la evidencia:")
    display(comp.reindex(comp["dif (emp. norm. − supuesta)"].abs().sort_values(ascending=False).index).head(10).round(2))

In [ ]:
if HAY_EVIDENCIA:
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
    for ax, M, tit, cmap, vmin, vmax in [(axes[0], SUP, "Supuesta (v1)", "Blues", 0, 1), (axes[1], EMP_N, "Empírica (avisos), normalizada por fila", "Blues", 0, 1),
                                          (axes[2], EMP_N - SUP, "Diferencia (empírica − supuesta)", "RdBu_r", -0.6, 0.6)]:
        im = ax.imshow(M.values, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
        ax.set_xticks(range(len(sim.CATS)), sim.CATS, rotation=45, ha="right", fontsize=7); ax.set_yticks(range(6), sim.OBJETIVOS, fontsize=8)
        for i_ in range(6):
            for j_ in range(len(sim.CATS)): ax.text(j_, i_, f"{M.values[i_, j_]:.1f}", ha="center", va="center", fontsize=6.5, color=TXT)
        ax.set_title(tit, fontsize=10)
    guardar(fig, "25_afinidad_supuesta_vs_empirica"); plt.show()

## 3. ¿Cambian las conclusiones si el simulador usa la matriz empírica?

Se repite el escenario base (E0 del notebook 06) con 5 datasets nuevos para cada matriz (supuesta y empírica) y se comparan los modelos con el corte global.

In [ ]:
if HAY_EVIDENCIA:
    P0 = sim.escenarios()["E0 Base (reglas v1)"]
    cursos = pd.read_csv(DATA / "cursos.csv")
    out = []
    for etiqueta, AF in [("Matriz supuesta (v1)", None), ("Matriz empírica (avisos)", EMP_N)]:
        u_aj, i_aj, _ = sim.generar_dataset(cursos, P0.con(seed=1000), afinidad=AF)
        hp = ex.ajustar_hiperparametros(cursos, u_aj, ex.construir_pares(i_aj), esquema="global")
        for r in range(5):
            u, i, tr = sim.generar_dataset(cursos, P0.con(seed=3000 + 10 * r), con_traza=True, afinidad=AF)
            _, res, n = ex.evaluar_dataset(cursos, u, i, tr, "global", hp)
            res["% del techo"] = 100 * res["ndcg"] / res.loc["Oráculo", "ndcg"]
            out.append(res.reset_index().assign(matriz=etiqueta, replica=r, evaluables=n))
    out = pd.concat(out)
    tab = out.groupby(["modelo", "matriz"])["% del techo"].agg(["mean", "std"]).unstack("matriz").round(1)
    tab.to_csv(RES / "afinidad_empirica_vs_supuesta.csv")
    display(tab.loc[["Popularidad", "Pop. por segmento + reglas", "Contenido (TF-IDF)", "KNN usuario", "KNN ítem", "LightFM híbrido", "LightFM + reglas"]])

## 4. Qué reportar

Cuando el archivo de evidencia exista, el informe puede incluir: (1) la tabla de avisos (fuente, fecha, cantidad por puesto); (2) la figura 25; (3) la correlación entre la matriz supuesta y la empírica; (4) la tabla de la sección 3. Si la correlación es alta y las conclusiones no cambian, la regla central del simulador queda respaldada por evidencia externa. Si no lo es, hay que **usar la matriz empírica como v2 del simulador** y volver a correr los notebooks 03 a 06.